# SinhalaCheck — Module 1: NLP Content Credibility Analysis
### Corrected-Data Retraining & Comparative Model Analysis

**Project:** R26-IT-158 | **Module 1** | Kaweeshwara P.D.S. (IT22331304) | SLIIT

---

### What this notebook does

1. **Documents a data integrity failure** in the published LIRNEasia corpus (the distributed
   `Corpus.csv` has had all Sinhala characters destroyed) and recovers the intact source.
2. **Retrains** the content credibility classifier on the corrected text, using an
   *identical* train/test split and seed to the original run, so the only variable is data quality.
3. **Compares several BERT-based models** (panel requirement), replicating the model-selection
   benchmark of *BERTifying Sinhala* (Dhananjaya et al., LREC 2022) on this new task.
4. **Establishes classical baselines** (majority class, TF-IDF+Naive Bayes, TF-IDF+SVM) and tests
   whether the transformer's advantage is **statistically significant** (McNemar's test).
5. **Reports a strict fake-news subgroup analysis** on the FALSE/PARTIAL documents, since the
   binary collapse otherwise conflates *uncertainty* with *falsehood*.

### Before you run
- Runtime → Change runtime type → **T4 GPU**
- Have `Corpus.xlsx` (the intact corpus) ready to upload when prompted.
- Runtime → **Run all**. Expect roughly 60–90 minutes depending on which models are enabled.

## 0 — Environment

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas openpyxl statsmodels matplotlib seaborn
print("Environment ready.")

In [ ]:
import os, re, json, random, warnings, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- reproducibility
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU.")

# ---------------------------------------------------------------- experiment config
# Identical to the ORIGINAL run so the comparison isolates data quality:
TEST_SIZE     = 0.2
MAX_LENGTH    = 256
EPOCHS        = 3
LEARNING_RATE = 2e-5
BATCH_SIZE    = 16

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Config locked: seed=%d, test_size=%.1f, max_len=%d, epochs=%d, lr=%g"
      % (SEED, TEST_SIZE, MAX_LENGTH, EPOCHS, LEARNING_RATE))

## 1 — Data integrity check

The LIRNEasia corpus is distributed as `Corpus.csv`. That published file has been through a
lossy encoding conversion: every Sinhala codepoint has been replaced with the literal character
`?`. No decoding recovers it — the information is gone from the file.

The cell below downloads the published CSV and compares it, row for row, against the intact
`Corpus.xlsx`. This comparison is itself a reportable finding.

In [ ]:
from google.colab import files

# --- the published (corrupted) release, straight from the official repository
CSV_URL = "https://raw.githubusercontent.com/LIRNEasia/MisinformationCorpusSinhala/main/Corpus.csv"
!wget -q -O Corpus_published.csv $CSV_URL
print("Downloaded published Corpus.csv")

# --- the intact source
if not os.path.exists("Corpus.xlsx"):
    print("\nUpload Corpus.xlsx (the intact corpus):")
    up = files.upload()
    src = list(up.keys())[0]
    if src != "Corpus.xlsx":
        os.rename(src, "Corpus.xlsx")
print("Corpus.xlsx present.")

In [ ]:
SINHALA_RE = re.compile(r"[඀-෿]")

df_pub   = pd.read_csv("Corpus_published.csv", encoding="latin-1")   # only encoding that loads it
df_clean = pd.read_excel("Corpus.xlsx")

# ---- verify the two files describe the SAME documents in the SAME order.
aligned = (
    len(df_pub) == len(df_clean)
    and (df_pub["X1"].values == df_clean["X1"].values).all()
    and (df_pub["domain"].astype(str).values == df_clean["domain"].astype(str).values).all()
)
print("Row-for-row aligned:", aligned)
assert aligned, "Files are not aligned - the ablation below would not be valid."

def sinhala_chars(s):  return len(SINHALA_RE.findall(str(s)))
def qmark_chars(s):    return str(s).count("?")

integrity = pd.DataFrame({
    "file":                 ["Corpus.csv (published)", "Corpus.xlsx (intact)"],
    "documents":            [len(df_pub), len(df_clean)],
    "mean Sinhala chars/doc": [df_pub["content"].apply(sinhala_chars).mean(),
                               df_clean["content"].apply(sinhala_chars).mean()],
    "mean '?' chars/doc":     [df_pub["content"].apply(qmark_chars).mean(),
                               df_clean["content"].apply(qmark_chars).mean()],
})
integrity[["mean Sinhala chars/doc", "mean '?' chars/doc"]] = \
    integrity[["mean Sinhala chars/doc", "mean '?' chars/doc"]].round(1)

print("\n" + "="*74)
print("DATA INTEGRITY COMPARISON")
print("="*74)
print(integrity.to_string(index=False))

print("\nSame document, both files:")
print("  published :", str(df_pub['content'].iloc[0])[:90])
print("  intact    :", str(df_clean['content'].iloc[0])[:90])

integrity.to_csv(f"{RESULTS_DIR}/data_integrity.csv", index=False)

## 2 — Labels

Two things to handle carefully:

**(a) Excel silently coerced the string `"FALSE"` into a boolean.** Reading the sheet naively
yields a `type` column containing `False` (bool) alongside string labels, and any string
comparison then drops those 27 documents without warning.

**(b) The binary collapse.** `CREDIBLE = 1`, everything else `= 0`. This is the contract the
rest of the system codes against, so it is what we ship. But note what "everything else" is:
overwhelmingly `UNCERTAIN`, not `FALSE`. We keep the original four-way label alongside the
binary one so we can report the strict fake-news subgroup separately in section 8.

In [ ]:
def normalise_type(v):
    # Excel turned the string "FALSE" into a boolean - repair it before comparing.
    if isinstance(v, bool):
        return "FALSE" if v is False else "TRUE"
    return str(v).strip().upper()

data = pd.DataFrame({
    "text_clean":     df_clean["content"].astype(str),
    "text_published": df_pub["content"].astype(str),   # the corrupted version of the SAME rows
    "type":           df_clean["type"].apply(normalise_type),
    "domain":         df_clean["domain"].astype(str),
})
data = data[data["text_clean"].str.strip().str.len() > 0].reset_index(drop=True)
data["label"] = (data["type"] == "CREDIBLE").astype(int)

print("Four-way label distribution:")
print(data["type"].value_counts().to_string())
print("\nBinary distribution (1 = CREDIBLE):")
print(data["label"].value_counts().to_string())
print(f"\nTotal documents: {len(data)}")

n_strict = int(data['type'].isin(['FALSE','PARTIAL']).sum())
print(f"\nNote: documents labelled FALSE or PARTIAL: {n_strict} "
      f"({100*n_strict/len(data):.1f}% of corpus)")
print("      -> the NOT-CREDIBLE class is dominated by UNCERTAIN. See section 8.")

## 3 — Split

Stratified 80/20, `random_state=42` — identical to the original run. We split **indices** rather
than texts, so the corrupted and corrected versions of every document land in the same fold, and
so we can recover each test document's original four-way label later.

In [ ]:
from sklearn.model_selection import train_test_split

idx_train, idx_test = train_test_split(
    np.arange(len(data)),
    test_size=TEST_SIZE, random_state=SEED, stratify=data["label"].values
)

train_df = data.iloc[idx_train].reset_index(drop=True)
test_df  = data.iloc[idx_test].reset_index(drop=True)

y_train = train_df["label"].values
y_test  = test_df["label"].values

print(f"Train: {len(train_df)}   Test: {len(test_df)}")
print(f"Test set composition: {(y_test==0).sum()} NOT CREDIBLE / {(y_test==1).sum()} CREDIBLE")

MAJORITY = int(np.bincount(y_train).argmax())
majority_acc = (y_test == MAJORITY).mean()
print(f"\n>>> Majority-class accuracy on this test set: {majority_acc:.4f}")
print("    Any model that does not clearly exceed this has learned nothing.")

## 4 — Evaluation helpers

Every model in this notebook is scored the same way on the same test set, and we retain each
model's **per-document predictions** so that McNemar's test can compare them pairwise later.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

RESULTS     = {}   # name -> metrics dict
PREDICTIONS = {}   # name -> np.array of predictions on the test set

def record(name, y_true, y_pred, note=""):
    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    wtd_f1   = f1_score(y_true, y_pred, average="weighted")
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average="binary", zero_division=0)
    RESULTS[name] = {
        "model": name, "note": note,
        "accuracy": acc, "macro_f1": macro_f1, "weighted_f1": wtd_f1,
        "credible_precision": p, "credible_recall": r, "credible_f1": f,
    }
    PREDICTIONS[name] = np.asarray(y_pred)
    print(f"\n{'='*74}\n{name}  {note}\n{'='*74}")
    print(f"Accuracy {acc:.4f} | Macro-F1 {macro_f1:.4f} | Weighted-F1 {wtd_f1:.4f}")
    print(f"CREDIBLE  precision {p:.3f}  recall {r:.3f}  f1 {f:.3f}")
    print(classification_report(y_true, y_pred,
          target_names=["NOT CREDIBLE", "CREDIBLE"], zero_division=0))
    return RESULTS[name]

## 5 — Classical baselines

These answer *"compared to what?"*. The TF-IDF + SVM baseline in particular is the approach used
elsewhere in this project, so beating it is a comparison against a real, in-project alternative
rather than a number quoted from a paper.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# ---- 5.1 Majority class
record("Majority class", y_test, np.full_like(y_test, MAJORITY),
       note="(predicts the most frequent training label for every document)")

# ---- shared TF-IDF space, fit on TRAIN ONLY (no leakage)
tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
Xtr = tfidf.fit_transform(train_df["text_clean"])
Xte = tfidf.transform(test_df["text_clean"])
print(f"\nTF-IDF vocabulary: {len(tfidf.vocabulary_)} features")

# ---- 5.2 Naive Bayes  (the published Sinhala baseline family)
nb = MultinomialNB().fit(Xtr, y_train)
record("TF-IDF + Naive Bayes", y_test, nb.predict(Xte), note="(classical baseline)")

# ---- 5.3 Linear SVM
svm = LinearSVC(class_weight="balanced", random_state=SEED).fit(Xtr, y_train)
record("TF-IDF + Linear SVM", y_test, svm.predict(Xte), note="(classical baseline)")

## 6 — Transformer training

One function, used identically for every model, so the comparison is fair: same split, same
seed, same hyperparameters, same evaluation. Mixed precision keeps the larger models inside a
T4's memory budget.

Each model is wrapped in error handling — if one checkpoint fails to download or does not fit in
memory, the notebook records the failure and continues rather than losing the whole run.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length",
                             max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def train_and_evaluate(model_name, train_texts, y_tr, test_texts, y_te,
                       label, note="", batch_size=BATCH_SIZE, epochs=EPOCHS):
    # Fine-tune `model_name` and record its test-set performance. Returns predictions or None.
    print(f"\n{'#'*74}\n# {label}\n# checkpoint: {model_name}\n{'#'*74}")
    t0 = time.time()
    set_seed()   # every model starts from the same random state

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2,
            id2label={0: "NOT_CREDIBLE", 1: "CREDIBLE"},
            label2id={"NOT_CREDIBLE": 0, "CREDIBLE": 1},
        ).to(device)
    except Exception as e:
        print(f"!! Could not load {model_name}: {type(e).__name__}: {e}")
        return None

    train_loader = DataLoader(TextDataset(train_texts, y_tr, tokenizer),
                              batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TextDataset(test_texts,  y_te, tokenizer),
                              batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    try:
        for epoch in range(epochs):
            model.train()
            running, correct, seen = 0.0, 0, 0
            for step, batch in enumerate(train_loader):
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    out = model(**batch)
                    loss = out.loss
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update(); scheduler.step()

                running += loss.item()
                correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
                seen    += batch["labels"].size(0)
                if step % 25 == 0:
                    print(f"  epoch {epoch+1}/{epochs}  step {step:>4}/{len(train_loader)}  loss {loss.item():.4f}")
            print(f"  -> epoch {epoch+1} done | mean loss {running/len(train_loader):.4f} "
                  f"| train acc {100*correct/seen:.2f}%")
    except torch.cuda.OutOfMemoryError:
        print(f"!! Out of memory for {label}. Retry with a smaller batch_size.")
        del model, optimizer, scaler
        gc.collect(); torch.cuda.empty_cache()
        return None

    # ---- evaluate
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(input_ids=batch["input_ids"],
                           attention_mask=batch["attention_mask"]).logits
            preds.extend(logits.argmax(-1).cpu().numpy())
    preds = np.asarray(preds)

    record(label, y_te, preds, note=note)
    RESULTS[label]["checkpoint"]   = model_name
    RESULTS[label]["train_minutes"] = round((time.time() - t0) / 60, 1)
    print(f"  [{label} finished in {RESULTS[label]['train_minutes']} min]")

    # IMPORTANT: park the finished model on the CPU. Keeping every trained model
    # resident in VRAM is what starved the later (larger) models of memory.
    TRAINED[label] = (model.to("cpu"), tokenizer)
    del model, optimizer, scaler
    gc.collect(); torch.cuda.empty_cache()
    if device.type == "cuda":
        print(f"  [GPU memory now allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB]")
    return preds

TRAINED = {}
print("Training harness ready.")

### 6.1 — The control run: same model, corrupted text

This is the ablation that isolates the data-quality effect. Identical model, identical split,
identical hyperparameters — the *only* difference is that the input text is the published
(corrupted) version of the very same documents.

In [ ]:
_ = train_and_evaluate(
    "xlm-roberta-base",
    train_df["text_published"], y_train,
    test_df["text_published"],  y_test,
    label="XLM-R base [CORRUPTED data]",
    note="(control: published Corpus.csv - Sinhala destroyed)",
)

### 6.2 — Model comparison on corrected text

Candidates follow *BERTifying Sinhala* (Dhananjaya et al., LREC 2022), which benchmarked
pretrained models for Sinhala text classification and concluded that XLM-R-large is the strongest
choice where hardware permits, with SinBERT competitive on smaller datasets. We test that
finding on a new task — content credibility.

Disable any row in `MODEL_ZOO` if you are short on GPU time; XLM-R-large is the slowest by far.

In [ ]:
MODEL_ZOO = [
    # (checkpoint,                      display label,        run?, batch)
    ("xlm-roberta-base",                "XLM-R base",          True,  16),
    ("bert-base-multilingual-cased",    "mBERT",               True,  16),
    ("setu4993/LaBSE",                  "LaBSE",               True,   8),
    ("NLPC-UOM/SinBERT-small",          "SinBERT-small",       True,  16),
    ("NLPC-UOM/SinBERT-large",          "SinBERT-large",       True,   8),
    ("xlm-roberta-large",               "XLM-R large",         True,   4),
]

for ckpt, label, run, bs in MODEL_ZOO:
    if not run:
        print(f"(skipping {label})"); continue
    train_and_evaluate(
        ckpt,
        train_df["text_clean"], y_train,
        test_df["text_clean"],  y_test,
        label=label, note="(corrected data)", batch_size=bs,
    )

## 7 — Results table

In [ ]:
cols = ["model", "note", "accuracy", "macro_f1", "weighted_f1",
        "credible_precision", "credible_recall", "credible_f1"]
res = pd.DataFrame(RESULTS).T[cols].reset_index(drop=True)
for c in cols[2:]:
    res[c] = res[c].astype(float).round(4)
res = res.sort_values("macro_f1", ascending=False).reset_index(drop=True)

print("="*100)
print("COMPARATIVE RESULTS  -  identical test set (n=%d), identical split (seed=%d)" % (len(y_test), SEED))
print("="*100)
print(res.to_string(index=False))

res.to_csv(f"{RESULTS_DIR}/comparative_results.csv", index=False)
print(f"\nSaved -> {RESULTS_DIR}/comparative_results.csv")

best = res.iloc[0]["model"]
print(f"\n>>> Best by macro-F1: {best}")
print(f">>> Majority-class floor: {majority_acc:.4f} accuracy")

In [ ]:
import matplotlib.pyplot as plt

plot_df = res.sort_values("macro_f1")
fig, ax = plt.subplots(figsize=(9, 0.55*len(plot_df)+2))
colors = ["#c0392b" if "CORRUPTED" in m else
          "#7f8c8d" if ("Majority" in m or "TF-IDF" in m) else "#2e86c1"
          for m in plot_df["model"]]
ax.barh(plot_df["model"], plot_df["macro_f1"].astype(float), color=colors)
ax.axvline(f1_score(y_test, np.full_like(y_test, MAJORITY), average="macro"),
           ls="--", c="k", lw=1, label="majority-class floor")
ax.set_xlabel("Macro F1"); ax.set_title("Module 1 — comparative model performance")
ax.legend(); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/model_comparison.png", dpi=160)
plt.show()

## 8 — Statistical significance (McNemar's test)

An accuracy gap on 600 test documents is not, by itself, evidence. McNemar's test asks whether
two classifiers **disagree asymmetrically** on the same items — the correct test for comparing
two models evaluated on one shared test set.

- H₀: the two models have the same error rate.
- p < 0.05 ⇒ the difference is unlikely to be sampling noise.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(a, b):
    pa, pb = PREDICTIONS[a], PREDICTIONS[b]
    ca, cb = (pa == y_test), (pb == y_test)
    tbl = np.array([[np.sum(ca & cb),  np.sum(ca & ~cb)],
                    [np.sum(~ca & cb), np.sum(~ca & ~cb)]])
    # exact binomial test when discordant pairs are few
    n_disc = tbl[0,1] + tbl[1,0]
    r = mcnemar(tbl, exact=(n_disc < 25), correction=True)
    return {"model_A": a, "model_B": b,
            "A_correct_B_wrong": int(tbl[0,1]), "B_correct_A_wrong": int(tbl[1,0]),
            "statistic": float(r.statistic) if r.statistic is not None else np.nan,
            "p_value": float(r.pvalue),
            "significant_at_0.05": bool(r.pvalue < 0.05)}

best_model = res.iloc[0]["model"]
comparisons = [m for m in PREDICTIONS if m != best_model]

sig = pd.DataFrame([mcnemar_test(best_model, m) for m in comparisons])
sig["p_value"] = sig["p_value"].apply(lambda p: f"{p:.2e}" if p < 1e-3 else f"{p:.4f}")

print("="*100)
print(f"McNEMAR'S TEST  -  '{best_model}' vs every other model")
print("="*100)
print(sig.to_string(index=False))
sig.to_csv(f"{RESULTS_DIR}/significance_tests.csv", index=False)

print("\nReading this table: 'A_correct_B_wrong' counts documents the best model got right and")
print("the comparison model got wrong. A significant result means that asymmetry is real,")
print("not an artefact of which documents happened to land in the test split.")

## 9 — Strict fake-news subgroup analysis

The shipped classifier is binary, because that is the system-wide contract. But the
NOT-CREDIBLE class is overwhelmingly `UNCERTAIN`, so headline accuracy mostly measures
*credible vs uncertain*, not *credible vs false*.

This section reports, separately, how the model performs on the documents that are actually
labelled `FALSE` or `PARTIAL` — the genuine misinformation. Reporting this openly is stronger
than being asked about it.

In [ ]:
strict_mask = test_df["type"].isin(["FALSE", "PARTIAL"]).values
n_strict = int(strict_mask.sum())
print(f"Test documents labelled FALSE or PARTIAL: {n_strict} of {len(test_df)}")
print("Breakdown:", test_df.loc[strict_mask, 'type'].value_counts().to_dict())

if n_strict > 0:
    rows = []
    for name, preds in PREDICTIONS.items():
        caught = int((preds[strict_mask] == 0).sum())     # correctly flagged NOT CREDIBLE
        # Counterweight: a model that flags EVERYTHING scores 1.0 above and is useless.
        # Pair detection with recall on genuinely CREDIBLE documents.
        cred_mask = (y_test == 1)
        kept = float((preds[cred_mask] == 1).mean()) if cred_mask.sum() else float("nan")
        rows.append({"model": name,
                     "FALSE/PARTIAL docs": n_strict,
                     "correctly flagged": caught,
                     "detection rate": round(caught / n_strict, 4),
                     "CREDIBLE recall": round(kept, 4),
                     "balanced": round((caught / n_strict + kept) / 2, 4)})
    strict = pd.DataFrame(rows).sort_values("balanced", ascending=False)
    print("\n" + "="*80)
    print("DETECTION RATE ON GENUINE MISINFORMATION (FALSE / PARTIAL only)")
    print("="*80)
    print(strict.to_string(index=False))
    strict.to_csv(f"{RESULTS_DIR}/strict_subgroup.csv", index=False)
    print("\nRead 'detection rate' and 'CREDIBLE recall' TOGETHER. A model that labels every")
    print("document NOT CREDIBLE scores 1.0 detection and 0.0 CREDIBLE recall - it is useless.")
    print("The 'balanced' column is the mean of the two and is the honest ranking.")
    print("\nCaveat to state explicitly: n is small, so these rates carry wide confidence")
    print("intervals. They are reported for transparency about what the binary task measures,")
    print("not as a headline result.")
else:
    print("No FALSE/PARTIAL documents in this test fold.")

## 10 — Confusion matrices

In [ ]:
import seaborn as sns

show = [m for m in ["XLM-R base [CORRUPTED data]", best_model] if m in PREDICTIONS]
fig, axes = plt.subplots(1, len(show), figsize=(5.5*len(show), 4.5))
if len(show) == 1: axes = [axes]
for ax, name in zip(axes, show):
    cm = confusion_matrix(y_test, PREDICTIONS[name])
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["NOT CRED", "CREDIBLE"], yticklabels=["NOT CRED", "CREDIBLE"])
    ax.set_title(f"{name}\nacc={RESULTS[name]['accuracy']:.3f}  macroF1={RESULTS[name]['macro_f1']:.3f}")
    ax.set_xlabel("predicted"); ax.set_ylabel("actual")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/confusion_matrices.png", dpi=160); plt.show()

## 11 — Summary to paste back into the chat

In [ ]:
print("="*100)
print("SINHALACHECK MODULE 1 - RUN SUMMARY")
print("="*100)
print(f"\nTest set: n={len(y_test)}  ({(y_test==0).sum()} NOT CREDIBLE / {(y_test==1).sum()} CREDIBLE)")
print(f"Majority-class floor: {majority_acc:.4f}\n")
print(res.to_string(index=False))
print("\n" + "-"*100)
print("SIGNIFICANCE (vs best model)")
print("-"*100)
print(sig.to_string(index=False))
if n_strict > 0:
    print("\n" + "-"*100); print("STRICT FAKE-NEWS SUBGROUP"); print("-"*100)
    print(strict.to_string(index=False))
print("\n" + "-"*100); print("DATA INTEGRITY"); print("-"*100)
print(integrity.to_string(index=False))
print("\n" + "="*100)
print("Copy everything above and paste it back into the chat.")
print("="*100)

## 12 — Save the selected model

Saves the best model in the exact layout the existing Flask app expects, so `app.py` picks it up
with no code change. `id2label` is written into the config this time, which removes the ambiguity
about which index means CREDIBLE.

In [ ]:
# Drive mounting fails on some accounts ("credential propagation was unsuccessful").
# That must never destroy a completed run, so every step here is non-fatal.
OUT = "/content/SinhalaCheck_model_v3"      # local fallback
drive_ok = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/SinhalaCheck_model_v3"
    drive_ok = True
    print("Drive mounted.")
except Exception as e:
    print(f"!! Drive mount failed ({type(e).__name__}). Saving locally to {OUT} instead.")
    print("   Files -> folder icon in the left sidebar -> download from there.")

os.makedirs(OUT, exist_ok=True)

if best_model in TRAINED:
    model, tokenizer = TRAINED[best_model]
    model.save_pretrained(OUT)
    tokenizer.save_pretrained(OUT)
    print(f"Saved '{best_model}' -> {OUT}")
else:
    print(f"'{best_model}' not held in memory; re-run its cell before saving.")

manifest = {
    "selected_model":  best_model,
    "checkpoint":      RESULTS.get(best_model, {}).get("checkpoint"),
    "seed":            SEED,
    "test_size":       TEST_SIZE,
    "max_length":      MAX_LENGTH,
    "epochs":          EPOCHS,
    "learning_rate":   LEARNING_RATE,
    "label_map":       {"0": "NOT_CREDIBLE", "1": "CREDIBLE"},
    "n_train":         int(len(train_df)),
    "n_test":          int(len(test_df)),
    "majority_accuracy": float(majority_acc),
    "results":         {k: {kk: (float(vv) if isinstance(vv, (int, float, np.floating)) else vv)
                            for kk, vv in v.items()} for k, v in RESULTS.items()},
}
with open(f"{RESULTS_DIR}/run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

if drive_ok:
    !cp -r $RESULTS_DIR /content/drive/MyDrive/SinhalaCheck_results
    print("Results copied to Drive -> SinhalaCheck_results")
else:
    # Zip the (small) results folder so it can be pulled out of the browser directly.
    !zip -qr sinhalacheck_results.zip $RESULTS_DIR
    print("Results zipped -> sinhalacheck_results.zip")
    try:
        from google.colab import files as _f
        _f.download("sinhalacheck_results.zip")
    except Exception as e:
        print(f"   (auto-download unavailable: {type(e).__name__}) "
              f"- grab it from the Files sidebar.")